# Projection and Predicate pushdown in Apache Parquet using Spark

In [1]:
%run ../start.py

Spark version: 3.5.0, Driver memory: 16g, Executor memory: 8g
Spark packages: io.delta:delta-spark_2.12:3.3.2,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.11.0
Spark extensions: io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions


In [2]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            62Gi        16Gi        21Gi       925Mi        24Gi        44Gi
Swap:          8.0Gi          0B       8.0Gi


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("stefanoleone992/fifa-23-complete-player-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1


In [4]:
!ls -la {path}

total 5799848
drwxr-xr-x. 1 jovyan users        314 Jun 29 19:26  .
drwxr-xr-x. 1 jovyan users          2 Jun 29 18:45  ..
-rw-r--r--. 1 jovyan users       5356 Jun 29 18:45  female_coaches.csv
-rw-r--r--. 1 jovyan users   94212088 Jun 29 18:45  female_players.csv
-rw-r--r--. 1 jovyan users    1685124 Jun 29 18:45 'female_players (legacy).csv'
-rw-r--r--. 1 jovyan users    2214250 Jun 29 18:45  female_teams.csv
-rw-r--r--. 1 jovyan users     132879 Jun 29 18:45  male_coaches.csv
-rw-r--r--. 1 jovyan users 5637100640 Jun 29 18:46  male_players.csv
-rw-r--r--. 1 jovyan users   90933390 Jun 29 18:45 'male_players (legacy).csv'
-rw-r--r--. 1 jovyan users  112744779 Jun 29 18:46  male_teams.csv
drwxr-xr-x. 1 jovyan users         24 Jul  1 13:04  parquet


In [27]:
spark

time: 1.28 ms (started: 2026-07-01 15:31:08 +00:00)


In [7]:
!echo "Number of files: $(find {path}/parquet/male_players/*.parquet | wc -l)"
!echo "Files: $(ls {path}/parquet/male_players/*.parquet)"

Number of files: 42
Files: /home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1/parquet/male_players/part-00000-1c88ac02-29e4-497f-832d-233a8c740da3-c000.snappy.parquet
/home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1/parquet/male_players/part-00001-1c88ac02-29e4-497f-832d-233a8c740da3-c000.snappy.parquet
/home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1/parquet/male_players/part-00002-1c88ac02-29e4-497f-832d-233a8c740da3-c000.snappy.parquet
/home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1/parquet/male_players/part-00003-1c88ac02-29e4-497f-832d-233a8c740da3-c000.snappy.parquet
/home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1/parquet/male_players/part-00004-1c88ac02-29e4-497f-832d-233a8c740da3-c000.snappy.parquet
/home/jovyan/.cache/kagglehub/datasets

In [8]:
csv_file = Path(path) / "male_players.csv"
parquet_output = Path(path) / "parquet" / "male_players"

In [9]:
#!rm -rf {parquet_output}

In [10]:
%load_ext autotime

time: 120 µs (started: 2026-07-01 15:15:12 +00:00)


In [11]:
if not parquet_output.is_dir():
    print(f"Convert CSV to parquet")
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(str(csv_file))
    )

    df.write.mode("overwrite").parquet(str(parquet_output))
 
else:
    print(f"Read parquet")
    
    df = (
        spark.read
        .parquet(str(parquet_output))
    )      

Read parquet
time: 659 ms (started: 2026-07-01 15:15:12 +00:00)


# Create a massive dataframe

In [12]:
humanize.intword(df.count())

'10.0 million'

time: 569 ms (started: 2026-07-01 15:15:13 +00:00)


In [13]:
df.explain()

== Physical Plan ==
FileScan parquet [player_id#0,player_url#1,fifa_version#2,fifa_update#3,fifa_update_date#4,short_name#5,long_name#6,player_positions#7,overall#8,potential#9,value_eur#10,wage_eur#11,age#12,dob#13,height_cm#14,weight_kg#15,league_id#16,league_name#17,league_level#18,club_team_id#19,club_name#20,club_position#21,club_jersey_number#22,club_loaned_from#23,... 86 more fields] Batched: false, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-co..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<player_id:int,player_url:string,fifa_version:int,fifa_update:int,fifa_update_date:date,sho...


time: 8.95 ms (started: 2026-07-01 15:15:13 +00:00)


# Predicate pushdown

In [14]:
df_all = spark.read.parquet(str(parquet_output))

df_select = (
    spark.read
    .parquet(str(parquet_output))
    .select("value_eur")
)

time: 142 ms (started: 2026-07-01 15:15:13 +00:00)


In [15]:
df_select_filter = (
    spark.read
    .parquet(str(parquet_output))
    .filter(F.col("value_eur") > 1000000)
    .select("value_eur")
)

time: 52.2 ms (started: 2026-07-01 15:15:14 +00:00)


In [16]:
humanize.intword(df_select_filter.count())

'3.7 million'

time: 308 ms (started: 2026-07-01 15:15:14 +00:00)


In [17]:
humanize.intword(df_select.count())

'10.0 million'

time: 88.8 ms (started: 2026-07-01 15:15:14 +00:00)


# Physical plan
See "PushedFilters:"

In [18]:
df_select_filter.explain("formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#787]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1/parquet/male_players]
PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#787]

(3) Filter [codegen id : 1]
Input [1]: [value_eur#787]
Condition : (isnotnull(value_eur#787) AND (value_eur#787 > 1000000))


time: 7.55 ms (started: 2026-07-01 15:15:14 +00:00)


In [19]:
df_select.explain("formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#565]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1/parquet/male_players]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#565]


time: 3.94 ms (started: 2026-07-01 15:15:14 +00:00)


In [20]:
humanize.intword(df_select_filter.count())

'3.7 million'

time: 145 ms (started: 2026-07-01 15:15:14 +00:00)


In [21]:
humanize.intword(df_select.count())

'10.0 million'

time: 74.4 ms (started: 2026-07-01 15:15:14 +00:00)


In [22]:
def benchmark(label, query):
    # Run once to reduce JVM / planning noise
    query.collect()

    start = time.perf_counter()
    result = query.collect()
    elapsed = time.perf_counter() - start

    print(f"{label}: {elapsed:.3f} s")
    #return result

time: 192 µs (started: 2026-07-01 15:15:14 +00:00)


In [23]:
benchmark("Only value_eur", df_select)

Only value_eur: 8.722 s
time: 19 s (started: 2026-07-01 15:15:14 +00:00)


In [26]:
df_select_filter.explain("formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#787]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/.cache/kagglehub/datasets/stefanoleone992/fifa-23-complete-player-dataset/versions/1/parquet/male_players]
PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#787]

(3) Filter [codegen id : 1]
Input [1]: [value_eur#787]
Condition : (isnotnull(value_eur#787) AND (value_eur#787 > 1000000))


time: 1.67 ms (started: 2026-07-01 15:17:29 +00:00)


In [25]:
benchmark("Filtered, with predicate pushdown", df_select_filter)

Filtered, with predicate pushdown: 3.703 s
time: 6.9 s (started: 2026-07-01 15:17:07 +00:00)
